# PyTorch Tensor Shapes and Broadcasting

by Andrés Muñoz-Jaramillo

Shape mismatches are the single most common error when adapting PyTorch models to new tasks. This notebook builds an explicit mental model of how PyTorch represents multi-dimensional data, what broadcasting does, and — critically — how broadcasting can silently produce the wrong answer.

**This notebook is self-contained.** It uses only synthetic tensors that mirror the real Surya shapes, so it can be run without downloading any data or weights.

**Audience:** researchers comfortable with linear algebra and Python, new to PyTorch's shape conventions.

In [ ]:
import torch
import numpy as np
from einops import rearrange

---
## 1. Tensors

A tensor is a generalization of the scalars, vectors, and matrices to more dimensions:

| Math object | Rank | Example shape |
|---|---|---|
| Scalar | 0 | `()` |
| Vector | 1 | `(N,)` |
| Matrix | 2 | `(M, N)` |
| 3-D array | 3 | `(M, N, P)` |
| *n*-D array | *n* | `(d₀, d₁, …, dₙ₋₁)` |

PyTorch tensors carry dtype, device, and gradient information alongside the numeric values. The `.shape` attribute is the most important thing you will inspect.

In [ ]:
scalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.randn(4, 5)

print("scalar:", scalar.shape)   # torch.Size([])
print("vector:", vector.shape)   # torch.Size([3])
print("matrix:", matrix.shape)   # torch.Size([4, 5])

### Common construction patterns

In [ ]:
# Zeros, ones, random — shape is given as positional arguments or a tuple
a = torch.zeros(3, 4)          # 3×4 of zeros
b = torch.ones(2, 3, 4)        # 2×3×4 of ones
c = torch.randn(5, 13, 4, 64, 64)  # random normal — Surya-like shape

print(a.shape, b.shape, c.shape)

### Shape, dtype, device

These three attributes describe *where* the tensor is, *how* values are stored, and *what* the layout is.

In [ ]:
x = torch.randn(2, 13, 4, 64, 64, dtype=torch.float32)

print("shape :", x.shape)          # torch.Size([2, 13, 4, 64, 64])
print("dtype :", x.dtype)          # torch.float32
print("device:", x.device)         # cpu  (or cuda:0 if on GPU)
print("ndim  :", x.ndim)           # 5
print("numel :", x.numel())        # total number of elements

---
## 2. PyTorch Dimension Conventions

### 2.1 The standard layout: B C H W

PyTorch's default convention for image-like tensors is:

```
dimension 0 → B   batch size    (how many independent samples)
dimension 1 → C   channels      (spectral bands, features, ...)
dimension 2 → H   height        (spatial y)
dimension 3 → W   width         (spatial x)
```

This is the shape that convolutional layers, normalization layers, and most PyTorch utilities expect.

### 2.2 Surya's layout: B C T H W

Surya adds a **time** dimension, giving a 5-D tensor:

```
B  — batch of observations          (varies, e.g. 4 during training)
C  — instrument channels            (13: 8 AIA wavelengths + 5 HMI)
T  — input timestamps               (4 by default; configurable)
H  — height of the solar image      (64 after patching)
W  — width of the solar image       (64 after patching)
```

A **single item** returned by the dataset has shape `[C, T, H, W]` (no batch dimension).  
A **batch** assembled by the DataLoader has shape `[B, C, T, H, W]`.

A batch is a subset of the training or validation data that is used to modify gradients in a single step.

In [ ]:
# Representative Surya shapes (using small spatial size for speed)
B, C, T, H, W = 4, 13, 4, 64, 64

single_item = torch.randn(C, T, H, W)      # what __getitem__ returns
batch       = torch.randn(B, C, T, H, W)   # what the DataLoader returns

print("single item:", single_item.shape)   # [13, 4, 64, 64]
print("batch      :", batch.shape)         # [4, 13, 4, 64, 64]

### 2.3 Adding and removing dimensions with `einops.rearrange`

`rearrange` from the [einops](https://einops.rocks/) library makes every shape
transformation **explicit and self-documenting**. The pattern string describes
the shape on each side of the arrow.  This makes debugging and sizd manipulation
singificantly easier.

ALWAYS use it instead of `unsqueeze` / `squeeze` / `view` / `reshape` becuase these
kind of statements are hard to understand and might be ambiguous.

In [ ]:
item = torch.randn(C, T, H, W)          # [13, 4, 64, 64]

# Insert a batch dimension at position 0
item_as_batch = rearrange(item, 'c t h w -> 1 c t h w')   # [1, 13, 4, 64, 64]
print("add batch dim:", item_as_batch.shape)

# Remove it — the pattern is explicit about the 1 being there
back = rearrange(item_as_batch, '1 c t h w -> c t h w')   # [13, 4, 64, 64]
print("remove batch :", back.shape)

### 2.4 Reshaping with `einops.rearrange`

`rearrange` handles all shape transformations — flattening groups of dimensions, splitting them, and reordering — in one explicit, readable call. The pattern string describes the shape on both sides of the arrow, so intent is always clear.

Avoid `reshape`, `view`, and `permute` for the same reason you avoid bare `squeeze`: the pattern string documents *which* axes are being moved and *why*, and raises immediately if the actual shape does not match.

In [ ]:
x = torch.randn(B, C, T, H, W)           # [4, 13, 4, 64, 64]

# Flatten time+channel and spatial dimensions for a 1-D readout head
x_flat = rearrange(x, 'b c t h w -> b (c t) (h w)')      # [4, 52, 4096]
print("flat  :", x_flat.shape)

# Flatten everything except the batch dimension
x_flat2 = rearrange(x, 'b c t h w -> b (c t h w)')        # [4, 13*4*64*64]
print("flat2 :", x_flat2.shape)

# Combine channel and time dimensions, but keep spatial dimensions separate
# (sometimes necessary to run convolutional layers over the spatial dimensions)
x_flat2 = rearrange(x, 'b c t h w -> b (c t) h w')        # [4, 13*4, 64, 64]
print("flat3 :", x_flat2.shape)


# Reorder to [B, T, C, H, W] (sometimes needed for time-series models)
x_permuted = rearrange(x, 'b c t h w -> b t c h w')       # [4, 4, 13, 64, 64]
print("permuted:", x_permuted.shape)

---
## 3. Broadcasting

Broadcasting is the mechanism that lets PyTorch apply operations between tensors of **different shapes** without explicitly copying data. It follows NumPy's rules:

> Two shapes are compatible if, **reading from the right**, each pair of dimensions is either **equal** or one of them is **1**.

If a dimension is 1 on one side, that tensor is logically repeated along that axis to match the other.

### 3.1 Visual alignment

Alignment always happens from the **right** (trailing dimension). Short shapes are implicitly padded with 1s on the left:

```
Shape A:   [4, 13,  4, 64, 64]    (batch of Surya stacks)
Shape B:        [13,  1,  1,  1]   → implicitly [1, 13,  1,  1,  1]
             ↑   ↑   ↑   ↑   ↑
Result :   [4, 13,  4, 64, 64]    ✓ each dim is equal or one is 1
```

This is exactly what happens when you apply a per-channel scale factor.

In [ ]:
batch  = torch.randn(B, C, T, H, W)           # [4, 13, 4, 64, 64]

# Per-channel scale: one value per channel, nothing else
scale  = torch.rand(C, 1, 1, 1)               # [13, 1, 1, 1]

scaled = batch * scale                         # [4, 13, 4, 64, 64]
print("batch * per-channel scale:", scaled.shape)

In [ ]:
# Verify the math: pixel [0,2,1,10,10] should equal batch[0,2,1,10,10] * scale[2,0,0,0]
idx_b, idx_c, idx_t, idx_h, idx_w = 0, 2, 1, 10, 10

expected = batch[idx_b, idx_c, idx_t, idx_h, idx_w] * scale[idx_c, 0, 0, 0]
actual   = scaled[idx_b, idx_c, idx_t, idx_h, idx_w]

print(f"expected: {expected.item():.6f}")
print(f"actual  : {actual.item():.6f}")
print(f"match   : {torch.isclose(expected, actual).item()}")

### 3.2 Broadcasting rules in plain language

1. **Pad from the left.** If shapes have different numbers of dimensions, prepend 1s to the shorter shape.
2. **Check compatibility.** Two sizes are compatible if they are equal, or if one of them is 1.
3. **Expand.** Dimensions of size 1 are *conceptually* expanded to match the other side. No data is copied; PyTorch tracks this with strides.
4. **If any dimension pair is neither equal nor 1, the operation raises an error.**

In [ ]:
# --- Compatible pairs (should succeed) ---

a = torch.randn(4, 13, 4, 64, 64)

print("scalar      :", (a + 1.0).shape)                      # () broadcasts everywhere
print("[1]         :", (a + torch.ones(1)).shape)            # [1] → [1,1,1,1,1]
print("[64]        :", (a + torch.ones(64)).shape)           # [64] aligns with W
print("[64,64]     :", (a + torch.ones(64, 64)).shape)       # aligns with H×W
print("[1,1,1,1,1] :", (a + torch.ones(1,1,1,1,1)).shape)   # all-1 shape

In [ ]:
# --- Incompatible pair (should raise) ---
# Shape [4, 13, 4, 64, 64] vs [13] — the [13] aligns with W=64, not C=13

try:
    result = a + torch.ones(13)    # [13] aligns to the W axis (size 64) → mismatch!
    print(result.shape)
except RuntimeError as e:
    print("RuntimeError:", e)

The example above shows why **alignment from the right matters**: `torch.ones(13)` aligns with `W=64`, not `C=13`, so it fails — as it should.  
To broadcast over channels you need `torch.ones(13, 1, 1, 1)` (or `torch.ones(1, 13, 1, 1, 1)` for batches).

In [ ]:
# Correct per-channel broadcast
channel_bias = torch.randn(1, 13, 1, 1, 1)        # explicit shape: [1, C, 1, 1, 1]
result = a + channel_bias                          # [4, 13, 4, 64, 64] ✓
print(result.shape)

### 3.3 Reduction operations and `keepdim`

Most errors involving broadcasting happen in combination with **reductions** (mean, std, sum). By default these collapse the reduced dimension, which can surprise you when you later try to subtract the result.

In [ ]:
x = torch.randn(B, C, T, H, W)    # [4, 13, 4, 64, 64]

# Mean over the spatial dimensions H and W
mu_collapsed = x.mean(dim=(-2, -1))            # [4, 13, 4]   ← H and W are gone
mu_kept      = x.mean(dim=(-2, -1), keepdim=True)  # [4, 13, 4, 1, 1] ← preserved as size-1

print("collapsed:", mu_collapsed.shape)
print("kept     :", mu_kept.shape)

# Subtraction with keepdim broadcasts cleanly
centered = x - mu_kept                        # [4, 13, 4, 64, 64] ✓
print("centered :", centered.shape)

In [ ]:
# Without keepdim you have to re-insert the dimensions manually.
# rearrange makes the axes explicit and the intent unambiguous.
centered_manual = x - rearrange(mu_collapsed, 'b c t -> b c t 1 1')
print("manual   :", centered_manual.shape)
print("identical:", torch.allclose(centered, centered_manual))

---
## 4. Broadcasting Silent Failures

This is the most dangerous section. Broadcasting only raises a `RuntimeError` when shapes are **incompatible**. When shapes happen to be **numerically compatible but semantically wrong**, PyTorch silently produces incorrect results.

The canonical trap: **a batch size of 1**.

### 4.1 Setup: the problematic scenario

You compute per-channel statistics over your batch. If your batch happens to have `B=1`, the result has shape `[1, C]` or `[1, C, 1, 1, 1]`. That looks fine — but when you subtract it from a batch with `B>1`, you get a result that *appears correct* but is applying the statistics of a single sample to all others.

In [ ]:
# Imagine computing per-channel mean over a batch of size 1
batch_of_one = torch.randn(1, C, T, H, W)          # [1, 13, 4, 64, 64]

# Mean over the batch dimension (dim=0)
channel_mean = batch_of_one.mean(dim=0, keepdim=True)   # [1, 13, 4, 64, 64]
print("channel_mean:", channel_mean.shape)

In [ ]:
# Now we apply this 'normalization' to a bigger batch
big_batch = torch.randn(B, C, T, H, W)              # [4, 13, 4, 64, 64]

normalized = big_batch - channel_mean               # [4, 13, 4, 64, 64] — NO ERROR!
print("normalized  :", normalized.shape)
print("\nNo error raised. But every sample is being centered on the statistics")
print("of a *single* unrelated observation. The result is wrong, silently.")

Why does this succeed? Because `[1, 13, 4, 64, 64]` is broadcast-compatible with `[4, 13, 4, 64, 64]` — the leading 1 expands to 4.

### 4.2 The batch=1 / channel confusion

An even subtler case: you accidentally mix up which axis is the batch and which is the channel when shapes align numerically.

In [ ]:
# You compute a per-sample summary: one value per batch item
# Intended shape: [B] = [4]
sample_score = torch.randn(4)     # e.g. predicted flare intensity per sample

# You want to weight the channel dimension of a [B, C] features tensor
features = torch.randn(4, 13)     # [B, C] = [4, 13]

# WRONG: sample_score [4] aligns with C=13? No — C=13 ≠ 4. This raises.
try:
    result = features * sample_score
    print(result.shape)
except RuntimeError as e:
    print("Good — caught:", e)

In [ ]:
# Correct: give sample_score an explicit channel axis so it aligns with the batch dim
result_correct = features * rearrange(sample_score, 'b -> b 1')
print("correct shape:", result_correct.shape)                # [4, 4]

# Now verify: result[i, :] = features[i, :] * sample_score[i]
for i in range(4):
    ok = torch.allclose(result_correct[i, :], features[i, :] * sample_score[i])
    print(f"  sample {i}: {ok}")

In [ ]:
# Correct: give sample_score an explicit channel axis so it aligns with the batch dim
result_correct = features * rearrange(sample_score, 'b -> b 1')
print("correct shape:", result_correct.shape)                # [4, 4]

# Now verify: result[i, :] = features[i, :] * sample_score[i]
for i in range(4):
    ok = torch.allclose(result_correct[i, :], features[i, :] * sample_score[i])
    print(f"  sample {i}: {ok}")

In [ ]:
# Your model outputs shape [B, 1] — one prediction per sample, singleton output dim
def model_output(batch_size):
    return torch.randn(batch_size, 1)

output_b4 = model_output(4)
output_b1 = model_output(1)

print("B=4, before squeeze:", output_b4.shape)   # [4, 1]
print("B=4, after squeeze: ", output_b4.squeeze().shape)  # [4]  ← correct
print()
print("B=1, before squeeze:", output_b1.shape)   # [1, 1]
print("B=1, after squeeze: ", output_b1.squeeze().shape)  # []   ← scalar! batch dim gone!

In [ ]:
# Safe alternative: einops.rearrange with an explicit pattern.
# The pattern 'b 1 -> b' will raise if the second dim is NOT exactly 1,
# so it documents AND enforces your intent in one call.
print("B=4, rearrange('b 1 -> b'):", rearrange(output_b4, 'b 1 -> b').shape)  # [4]   ✓
print("B=1, rearrange('b 1 -> b'):", rearrange(output_b1, 'b 1 -> b').shape)  # [1]   ✓  batch preserved

### 4.4 Catching silent failures with assertions

The best defense is to **make the expected shape explicit** immediately after every operation whose shape you care about. Python's `assert` is free at training time and catches the bug at the line that introduces it rather than several layers later.

In [ ]:
def safe_channel_normalize(x: torch.Tensor) -> torch.Tensor:
    """
    Normalize each channel of x independently across spatial and time dims.
    x: [B, C, T, H, W]
    returns: [B, C, T, H, W]
    """
    assert x.ndim == 5, f"Expected 5-D input [B,C,T,H,W], got {x.shape}"

    # Mean over T, H, W but NOT over B — keep per-sample statistics
    mu  = x.mean(dim=(2, 3, 4), keepdim=True)   # [B, C, 1, 1, 1]
    std = x.std (dim=(2, 3, 4), keepdim=True)   # [B, C, 1, 1, 1]

    assert mu.shape == (x.shape[0], x.shape[1], 1, 1, 1), \
        f"Unexpected mu shape: {mu.shape}"

    out = (x - mu) / (std + 1e-8)

    assert out.shape == x.shape, f"Output shape {out.shape} ≠ input shape {x.shape}"
    return out


batch = torch.randn(B, C, T, H, W)
normed = safe_channel_normalize(batch)
print("output shape:", normed.shape)

### 4.5 Summary of silent-failure patterns and their fixes

| Scenario | Dangerous code | Why it's wrong | Safe alternative |
|---|---|---|---|
| Subtract batch mean when B=1 | `x - x.mean(dim=0, keepdim=True)` applied to `x_new` | `[1,C,T,H,W]` broadcasts across any B | Compute mean over the full dataset, not a single batch; or assert `B > 1` |
| Per-sample weight on square dims | `[B,C] * score` when B==C | `[B]` aligns to C, not B | `rearrange(score, 'b -> b 1')` then multiply |
| Removing singleton dims | `.squeeze()` bare call | Removes **all** size-1 dims including B when B=1 | `rearrange(x, 'b 1 -> b')` — raises if dim is not exactly 1 |
| Channel stats from batch | `x.mean(dim=0)` | Averages across batch, leaving `[C,T,H,W]` | `x.mean(dim=(0,2,3,4), keepdim=True)` then assert shape |
| No guard at all | — | Bug surfaces far from its origin | `assert tensor.shape == expected` after every non-trivial rearrange/reduction |

---
## 5. Practical Exercises

Work through these to consolidate your understanding. Solutions are in the collapsed cells below each exercise.

### Exercise 1

You have a Surya batch `x` of shape `[B, C, T, H, W]` and a learned per-channel, per-timestamp weight vector `w` of shape `[C, T]`.

Write a one-liner that multiplies each `(c, t)` slice of `x` by `w[c, t]`, producing output of the same shape as `x`. Do **not** use a Python loop.

In [ ]:
x = torch.randn(B, C, T, H, W)
w = torch.rand(C, T)

# Your solution here
# result = ...

In [ ]:
# Solution
# rearrange w from [C, T] to [1, C, T, 1, 1] so it broadcasts over [B, C, T, H, W].
# The pattern makes every axis explicit — no counting unsqueeze calls.
result = x * rearrange(w, 'c t -> 1 c t 1 1')

assert result.shape == x.shape

# Verify one element
assert torch.isclose(result[0, 2, 1, 5, 5], x[0, 2, 1, 5, 5] * w[2, 1])
print("Exercise 1: correct", result.shape)

### Exercise 2

The following function is supposed to z-score normalize each **channel** across the entire batch (compute a single mean and std per channel pooled over B, T, H, W). It contains a subtle shape bug. Find and fix it.

In [ ]:
def buggy_channel_norm(x):
    # Intended: one mean per channel, averaged over B, T, H, W
    mu  = x.mean(dim=1, keepdim=True)   # <-- dim=1 is the channel dim!
    std = x.std (dim=1, keepdim=True)
    return (x - mu) / (std + 1e-8)

x = torch.randn(B, C, T, H, W)

# What does buggy_channel_norm actually compute?
out = buggy_channel_norm(x)
print("output shape:", out.shape)   # Same as input — no error raised!

# Spot the problem: print mu.shape when dim=1 is reduced
print("mu shape    :", x.mean(dim=1, keepdim=True).shape)  # [B, 1, T, H, W]
print("→ This normalizes pixel-wise ACROSS channels, not per-channel across samples.")

In [ ]:
# Fix: reduce over B, T, H, W — everything except the channel dimension
def fixed_channel_norm(x):
    # dim=(0, 2, 3, 4) = B, T, H, W — channel dim (1) is preserved
    mu  = x.mean(dim=(0, 2, 3, 4), keepdim=True)   # [1, C, 1, 1, 1]
    std = x.std (dim=(0, 2, 3, 4), keepdim=True)
    assert mu.shape == (1, C, 1, 1, 1), f"Unexpected mu shape: {mu.shape}"
    return (x - mu) / (std + 1e-8)

out = fixed_channel_norm(x)
print("output shape:", out.shape)   # [B, C, T, H, W]
print("Exercise 2: fixed")

### Exercise 3: spot the silent failure

The cell below runs without error for any batch size. But it is **only correct for B > 1**. Identify why, then write an assertion that would catch the silent failure when B=1.

In [ ]:
def apply_batch_residual(x: torch.Tensor, ref_batch: torch.Tensor) -> torch.Tensor:
    """Subtract the per-channel mean of ref_batch from x."""
    ref_mean = ref_batch.mean(dim=0, keepdim=True)  # [1, C, T, H, W]
    return x - ref_mean

ref = torch.randn(1, C, T, H, W)   # ref_batch with B=1
target = torch.randn(4, C, T, H, W)

out = apply_batch_residual(target, ref)
print("runs fine:", out.shape)

In [ ]:
# Solution explanation:
# When ref_batch has B=1, ref_mean == ref_batch (mean of one sample = that sample).
# This looks like proper centering but is actually subtracting one arbitrary sample
# from all targets. With B>1 you'd get the true multi-sample mean.
#
# The guard:

def safe_batch_residual(x: torch.Tensor, ref_batch: torch.Tensor) -> torch.Tensor:
    assert ref_batch.shape[0] > 1, (
        f"ref_batch must have B > 1 to compute a meaningful mean, got B={ref_batch.shape[0]}"
    )
    ref_mean = ref_batch.mean(dim=0, keepdim=True)
    return x - ref_mean

try:
    safe_batch_residual(target, ref)   # ref has B=1 → assertion fires
except AssertionError as e:
    print("Caught:", e)

# With a proper reference batch:
ref_proper = torch.randn(8, C, T, H, W)
out = safe_batch_residual(target, ref_proper)
print("With B=8 ref:", out.shape)

---
## 6. Key Takeaways

1. **Shape first, code second.** Before writing any operation, write down the shape you expect on a piece of paper (or a comment). Then verify with an assertion.

2. **Broadcasting aligns from the right.** A 1-D tensor of size N aligns with the *last* dimension. Use `rearrange` to move it to the dimension you actually intend.

3. **`keepdim=True` is almost always what you want** after a reduction that you plan to broadcast back against the original tensor.

4. **Never use bare `.squeeze()`** — use `rearrange(x, 'b 1 -> b')` instead. The pattern documents what you expect and raises immediately if the shape is wrong.

5. **B=1 is the most treacherous case.** Broadcasting a `[1, C, T, H, W]` quantity against a `[B, C, T, H, W]` batch succeeds for any B, but the semantics are only correct if that singleton represents a global statistic, not a single sample.

6. **`einops.rearrange` over `unsqueeze` / `view` / `reshape`.** The pattern string is executable documentation — it shows the shape before and after, and raises `EinopsError` if the actual shape does not match.